# Capitales Del Mundo: Del HTML Al DataFrame

![Mapa del mundo](https://upload.wikimedia.org/wikipedia/commons/8/80/World_map_-_low_resolution.svg)

En este cuaderno, el dataset protagonista es una tabla de capitales en Wikipedia.
Vas a convertir una pagina web en datos limpios listos para analizar.

# Ejercicio 1 (Guiado): Web Scraping Basico

Objetivo: extraer una tabla de Wikipedia, limpiarla y generar un primer mini-analisis visual.

Ruta de trabajo:
1. Descargar HTML
2. Parsear con BeautifulSoup
3. Encontrar tabla
4. Extraer filas
5. Crear DataFrame
6. Visualizar y guardar

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Paso 1: Descargar la pagina

In [ ]:
url = 'https://es.wikipedia.org/wiki/Anexo:Capitales_de_pa%C3%ADses'
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

print('Estado HTTP:', response.status_code)
print('Caracteres descargados:', len(response.text))

## Paso 2 y 3: Parsear HTML y localizar la tabla

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
tablas = soup.find_all('table', class_='wikitable')
print('Tablas wikitable encontradas:', len(tablas))

tabla = tablas[0]
cabeceras = [th.get_text(strip=True) for th in tabla.find('tr').find_all('th')]
cabeceras[:8]

## Paso 4 y 5: Extraer registros y construir DataFrame

In [ ]:
filas_datos = []
for fila in tabla.find_all('tr')[1:]:
    celdas = fila.find_all(['td', 'th'])
    if not celdas:
        continue

    fila_limpia = []
    for celda in celdas:
        texto = celda.get_text(' ', strip=True)
        texto = texto.split('[')[0].strip()
        fila_limpia.append(texto)

    filas_datos.append(fila_limpia)

df = pd.DataFrame(filas_datos, columns=cabeceras)
print('Shape:', df.shape)
df.head()

## Paso 6: Visualización exploratoria rápida del dataset

Para visualizar algo sencillo, contamos iniciales del pais (primera letra de la primera columna).

In [ ]:
col_pais = df.columns[0]
iniciales = df[col_pais].astype(str).str[0].str.upper().value_counts().head(10)

ax = iniciales.sort_values().plot(kind='barh', color='#2a9d8f')
ax.set_title('Top 10 iniciales de paises (muestra visual)')
ax.set_xlabel('Frecuencia')
ax.set_ylabel('Inicial')
plt.show()

In [ ]:
df.to_csv('capitales_paises.csv', index=False, encoding='utf-8')
df.to_excel('capitales_paises.xlsx', index=False)
print('Archivos generados: capitales_paises.csv y capitales_paises.xlsx')

## Checklist final
- [ ] He entendido el flujo descargar -> parsear -> extraer
- [ ] He generado DataFrame limpio
- [ ] He creado al menos una visualizacion
- [ ] He exportado resultados